---

## Step 1 — Individual Model Generation (Random Subspace + Forward Stepwise)

Reads molecular descriptors from a training set CSV file and generates multiple Ordinary Least Squares (OLS) linear regression models using a **random subspace strategy**:

1. Descriptors with near-zero variance are removed.
2. Random subsets of descriptors are sampled; highly correlated descriptors within each subset are dropped.
3. Forward stepwise selection (by p-value) identifies the best descriptors within each subset.
4. OLS models are fitted; duplicate descriptor combinations are discarded.

**Outputs:** best-descriptor table (CSV), model coefficients (CSV), info log (TXT).


In [ ]:
print("=" * 70)
print("STEP 1 — Individual model generation")
print("=" * 70)

# ----- USER CONFIGURATION ----------------------------------------------------
TARGET_NAME_S1    = "CDIFF"       # Molecular target identifier (used in output filenames)
MODEL_TYPE_S1     = "OLS"               # Model type label (informational; used in filenames)
DESCRIPTOR_CALC_S1 = "Mordred"  # Descriptor calculation software (informational)

from pathlib import Path
WORKING_DIR_S1        = Path("/Users/francocaram/Library/CloudStorage/GoogleDrive-francocaram8@gmail.com/My Drive/DOCTORADO/Cdifficile_ML/Data_sets")   # Directory containing all input files
TRAINING_FILE_S1      = "training_set.csv"                  # Training set CSV (must include 'NAME' and 'clase' columns)

VARIANCE_THRESHOLD_S1 = 0.05    # Descriptors with variance below this value are removed
                                 # (0.05 removes descriptors constant in ~95% of molecules)
NUM_SUBSETS_S1        = 100   # Number of random descriptor subsets to generate
DESCRIPTORS_PER_SUBSET_S1 = 200 # Descriptors sampled per subset before correlation filtering
CORRELATION_METHOD_S1 = "pearson"  # Correlation coefficient: "pearson", "kendall", or "spearman"
CORRELATION_LIMIT_S1  = 0.8     # Pairs with |r| > this value: one descriptor is dropped
                                 # (0.5 = conservative; 0.9 = lenient)
SIGNIFICANCE_LEVEL_S1 = 0.05   # p-value threshold for forward stepwise descriptor inclusion
MAX_DESCRIPTORS_S1    = 5       # Maximum number of descriptors allowed per model
# ----- END CONFIGURATION -----------------------------------------------------

import sys
import statistics
from datetime import date

import numpy as np
import pandas as pd
import sklearn
import statsmodels.api as sm
from sklearn.feature_selection import VarianceThreshold
from sklearn.linear_model import LinearRegression
from sklearn.metrics import accuracy_score
from rdkit import rdBase

directorio_s1 = str(WORKING_DIR_S1)

# Load training data and drop any descriptor column containing at least one NaN
raw_data_s1 = pd.read_csv(directorio_s1 + "/" + TRAINING_FILE_S1, sep=",",
                           header="infer", index_col=None)
data_s1 = raw_data_s1.dropna(axis=1)

all_columns_s1      = list(data_s1)
name_class_s1       = data_s1[["NAME", "clase"]]
descriptor_names_s1 = all_columns_s1[2:]          # Columns 0 and 1 are NAME and clase
descriptors_s1      = data_s1[descriptor_names_s1]

# --- Step 1a: Variance filter ---
var_selector_s1 = VarianceThreshold(VARIANCE_THRESHOLD_S1)
var_selector_s1.fit(descriptors_s1)
descriptors_filtered_s1 = descriptors_s1[
    descriptors_s1.columns[var_selector_s1.get_support(indices=True)]
]

# --- Step 1b: Random subspace generation with correlation filtering ---
subsets_s1 = []
idx = 0
while idx < NUM_SUBSETS_S1:
    subset = descriptors_filtered_s1.sample(DESCRIPTORS_PER_SUBSET_S1, axis=1,
                                             random_state=idx)
    corr_matrix = subset.corr(CORRELATION_METHOD_S1).abs()
    upper_triangle = corr_matrix.where(
        np.triu(np.ones(corr_matrix.shape), k=1).astype(bool)
    )
    cols_to_drop = [
        col for col in upper_triangle.columns
        if any(upper_triangle[col] > CORRELATION_LIMIT_S1)
    ]
    cleaned_subset = subset.drop(cols_to_drop, axis=1)
    subsets_s1.append(pd.concat([name_class_s1, cleaned_subset], axis=1))
    idx += 1

# --- Step 1c: Forward stepwise OLS descriptor selection ---
def forward_stepwise_s1(X, y, significance_level=0.05, max_features=5):
    """
    Forward stepwise selection based on OLS p-values.
    Adds one descriptor at a time (the one with the lowest p-value) as long as
    its p-value is below `significance_level` and the total count is below
    `max_features`. Modifies the module-level list `selected_features_s1`.
    """
    all_features = X.columns.tolist()
    while len(all_features) > 0:
        remaining = list(set(all_features) - set(selected_features_s1))
        pvals = pd.Series(index=remaining, dtype=float)
        for feat in remaining:
            ols = sm.OLS(
                y,
                sm.add_constant(X[selected_features_s1 + [feat]])
            ).fit()
            pvals[feat] = ols.pvalues[feat]
        min_pval = pvals.min()
        if min_pval < significance_level and len(selected_features_s1) < max_features:
            selected_features_s1.append(pvals.idxmin())
        else:
            break

# --- Step 1d: Model fitting, deduplication, and evaluation on training set ---
model_counter_s1     = 1
model_names_s1       = []
fitted_models_s1     = []
model_coefficients_s1 = []
model_intercepts_s1  = []
descriptor_sets_s1   = []    
training_metrics_s1  = {}

for subset in subsets_s1:
    model_id = "MODEL_" + str(model_counter_s1)
    selected_features_s1 = []

    subset_columns    = list(subset)
    subset_desc_names = subset_columns[2:]
    X_subset = subset[subset_desc_names]
    y_subset = subset["clase"]

    forward_stepwise_s1(X_subset, y_subset,
                        SIGNIFICANCE_LEVEL_S1, MAX_DESCRIPTORS_S1)

    if selected_features_s1 not in descriptor_sets_s1:
        X_final = subset[selected_features_s1]
        ols_model = LinearRegression(fit_intercept=True, copy_X=True, n_jobs=None)
        try:
            ols_model.fit(X_final, y_subset)
            fitted_models_s1.append(ols_model)
            model_coefficients_s1.append(ols_model.coef_)
            model_intercepts_s1.append(ols_model.intercept_)

            training_scores = list(ols_model.predict(X_final))
            predicted_classes = pd.Series([1 if v > 0.5 else 0 for v in training_scores])
            df_by_class = pd.concat([y_subset, predicted_classes], axis=1)
            actives_df   = df_by_class[df_by_class["clase"] == 1]
            inactives_df = df_by_class[df_by_class["clase"] == 0]
            global_acc   = sum(y_subset.eq(predicted_classes)) / len(y_subset)
            acc_actives   = accuracy_score(actives_df["clase"],   actives_df[0])
            acc_inactives = accuracy_score(inactives_df["clase"], inactives_df[0])

            descriptor_sets_s1.append(selected_features_s1)
            model_names_s1.append(model_id)
            training_metrics_s1[model_counter_s1] = (
                model_id, global_acc, acc_actives, acc_inactives
            )
        except Exception:
            print(f"  [WARNING] Empty model at index {model_counter_s1} — skipped.")
    else:
        print(f"  [INFO] Duplicate descriptor set at index {model_counter_s1} — skipped.")

    model_counter_s1 += 1

# Save training accuracy table
metrics_df_s1 = pd.DataFrame.from_dict(training_metrics_s1, orient="index")
metrics_df_s1 = metrics_df_s1.rename(columns={
    0: "MODEL", 1: "Global_Acc_training", 2: "Acc_actives", 3: "Acc_inactives"
})
metrics_df_s1.to_csv(
    directorio_s1 + "/" + "training_results_" + MODEL_TYPE_S1 + "_" + TARGET_NAME_S1 + ".csv",
    sep=",", index=False
)

# Save best-descriptor table and full model table (coefficients + intercepts)
desc_table_s1 = pd.DataFrame(descriptor_sets_s1)
desc_table_s1.index = model_names_s1
desc_table_s1.to_csv(
    directorio_s1 + "/" + "best_features_" + MODEL_TYPE_S1 + "_" + TARGET_NAME_S1 + ".csv",
    sep=",", index=True
)

models_df_s1     = pd.DataFrame(model_names_s1, columns=["MODEL"])
intercept_df_s1  = pd.DataFrame(model_intercepts_s1, columns=["Intercept"])
coef_df_s1       = pd.DataFrame(model_coefficients_s1)
full_model_table_s1 = pd.concat([models_df_s1, intercept_df_s1, desc_table_s1, coef_df_s1], axis=1)
full_model_table_s1.to_csv(
    directorio_s1 + "/" + "models_" + MODEL_TYPE_S1 + "_" + TARGET_NAME_S1 + ".csv",
    sep=",", index=False
)

# Save run-information log
subset_sizes_s1 = [len(s.columns) - 2 for s in subsets_s1]
today_s1 = date.today().strftime("%d/%m/%Y")

with open(directorio_s1 + "/" + "model_info.txt", "w") as log:
    log.write("Model generation log — " + today_s1 + "\n\n")
    log.write("Descriptor calculation software  : " + DESCRIPTOR_CALC_S1 + "\n")
    log.write("Initial descriptor count         : " + str(len(raw_data_s1.columns) - 2) + "\n")
    log.write("After NaN removal                : " + str(len(data_s1.columns) - 2) + "\n")
    log.write("Variance threshold               : " + str(VARIANCE_THRESHOLD_S1) + "\n")
    log.write("After variance filter            : " + str(len(descriptors_filtered_s1.columns)) + "\n\n")
    log.write("Number of random subsets         : " + str(NUM_SUBSETS_S1) + "\n")
    log.write("Descriptors per subset           : " + str(DESCRIPTORS_PER_SUBSET_S1) + "\n")
    log.write("Correlation method               : " + CORRELATION_METHOD_S1 + "\n")
    log.write("Correlation cutoff               : " + str(CORRELATION_LIMIT_S1) + "\n")
    log.write("Mean descriptors after filtering : "
              + str(round(statistics.mean(subset_sizes_s1), 2))
              + " ± " + str(round(statistics.stdev(subset_sizes_s1), 2)) + "\n\n")
    log.write("Forward stepwise p-value cutoff  : " + str(SIGNIFICANCE_LEVEL_S1) + "\n")
    log.write("Maximum descriptors per model    : " + str(MAX_DESCRIPTORS_S1) + "\n\n")
    log.write("Python version    : " + sys.version + "\n")
    log.write("scikit-learn      : " + sklearn.__version__ + "\n")
    log.write("statsmodels       : " + sm.__version__ + "\n")
    log.write("RDKit             : " + rdBase.rdkitVersion + "\n")

print("Step 1 complete.\n")


---

## Step 2 — Evaluation of Individual Models on External Validation Sets

Loads the best-descriptor file produced in Step 1 and evaluates each model on external validation sets. For each set, computes:

- **AUC-ROC** — Area Under the Receiver Operating Characteristic curve
- **AUPR** — Area Under the Precision-Recall curve

After processing all sets, per-set standard deviation (SD) columns are appended, summarising the spread of AUC and AUPR across individual models within that validation set (sample SD, ddof=1).

**Outputs:** per-compound score tables (CSV) and a summary metrics table (CSV).


In [ ]:
print("=" * 70)
print("STEP 2 — Individual model evaluation on validation sets")
print("=" * 70)

# ----- USER CONFIGURATION ----------------------------------------------------
TARGET_NAME_S2         = "CDIFF"
MODEL_TYPE_S2          = "OLS"
WORKING_DIR_S2         = Path("/Users/francocaram/Library/CloudStorage/GoogleDrive-francocaram8@gmail.com/My Drive/DOCTORADO/Cdifficile_ML/Data_sets")
TRAINING_FILE_S2       = "training_set.csv"
BEST_FEATURES_FILE_S2  = "best_features_OLS_CDIFF.csv"  # Output from Step 1
VALIDATION_FILES_S2    = ["test_set_1.csv", "test_set_2.csv"]
# ----- END CONFIGURATION -----------------------------------------------------

from sklearn.metrics import roc_auc_score, precision_recall_curve, average_precision_score

directorio_s2 = str(WORKING_DIR_S2)

# Load best-descriptor file
feat_file_s2 = open(directorio_s2 + "/" + BEST_FEATURES_FILE_S2, "r")
next(feat_file_s2)
best_features_s2 = []
for line in feat_file_s2:
    parts = line.strip().split(",")
    best_features_s2.append(list(filter(None, parts[1:])))

model_id_table_s2 = pd.read_csv(directorio_s2 + "/" + BEST_FEATURES_FILE_S2, sep=",")
model_id_list_s2  = model_id_table_s2["Unnamed: 0"].tolist()

# Load and clean training data
training_data_s2 = pd.read_csv(directorio_s2 + "/" + TRAINING_FILE_S2, sep=",",
                                header="infer", index_col=None)
training_data_s2 = training_data_s2.dropna(axis=1)

results_header_s2 = ["MODEL"]
all_results_s2    = []
seen_models_s2    = []
val_index_s2      = 1

# Accumulators for per-set metric vectors (one value per model), used later for SD
auc_por_set_s2    = []   # auc_por_set_s2[i]    -> AUC    list for validation i+1
aupr_por_set_s2   = []   # aupr_por_set_s2[i]   -> AUPR   list for validation i+1

for val_file in VALIDATION_FILES_S2:
    val_data = pd.read_csv(directorio_s2 + "/" + val_file, sep=",",
                           header="infer", index_col=None)
    val_data.dropna(axis=1, inplace=True, how="all")

    val_label    = "val" + str(val_index_s2)
    model_ids    = []
    auc_list     = []
    aupr_list    = []
    scores_list  = []

    for i, feats in enumerate(best_features_s2):
        mid = model_id_list_s2[i]
        if mid not in seen_models_s2:
            seen_models_s2.append(mid)

        # Fit model on full training set
        X_train = training_data_s2[feats]
        y_train = training_data_s2["clase"]
        reg = LinearRegression(fit_intercept=True, copy_X=True, n_jobs=None)
        reg.fit(X_train, y_train)

        # Evaluate on validation set
        val_subset = pd.concat([val_data["clase"], val_data[feats]], axis=1)
        val_subset.fillna(val_subset[val_subset["clase"] == 0].mean(), inplace=True)
        X_val = val_subset[feats]
        y_val = val_subset["clase"]

        val_scores = list(reg.predict(X_val))
        scores_list.append(val_scores)
        auc_list.append(roc_auc_score(y_val, val_scores))
        aupr_list.append(average_precision_score(y_val, val_scores))

        model_ids.append(mid)

    # Save per-compound score table
    scores_df = pd.DataFrame(scores_list).T
    scores_df.columns = model_ids
    scores_full = pd.concat([val_data["NAME"], val_data["clase"], scores_df], axis=1)
    scores_full.to_csv(
        directorio_s2 + "/" + "individual_scores_" + val_label + "_"
        + MODEL_TYPE_S2 + "_" + TARGET_NAME_S2 + ".csv",
        sep=",", index=False
    )

    all_results_s2.append(auc_list)
    all_results_s2.append(aupr_list)
    results_header_s2.append("AUC_" + val_label)
    results_header_s2.append("AUPR_"   + val_label)
    # Store metric vectors for SD calculation
    auc_por_set_s2.append(auc_list)
    aupr_por_set_s2.append(aupr_list)
    val_index_s2 += 1

# Build summary metrics table
model_name_col_s2 = pd.read_csv(directorio_s2 + "/" + BEST_FEATURES_FILE_S2, sep=",")[["Unnamed: 0"]]
results_matrix_s2 = pd.DataFrame(all_results_s2).T
summary_s2 = pd.concat([model_name_col_s2, results_matrix_s2], axis=1)
summary_s2.columns = results_header_s2

# Add per-set SD columns (sample SD, ddof=1) — one SD per metric per validation set
# Each SD value is constant across rows: it summarises the spread of that metric
# across all individual models within that validation set.
for idx_sd in range(len(VALIDATION_FILES_S2)):
    val_label_sd = "val" + str(idx_sd + 1)
    summary_s2["SD_AUC_" + val_label_sd] = np.std(auc_por_set_s2[idx_sd], ddof=1)
    summary_s2["SD_AUPR_"   + val_label_sd] = np.std(aupr_por_set_s2[idx_sd],   ddof=1)
summary_s2.to_csv(
    directorio_s2 + "/" + "individual_results_" + MODEL_TYPE_S2 + "_" + TARGET_NAME_S2 + ".csv",
    sep=",", index=False
)

print("Step 2 complete.\n")


---

## Step 3 — Non-Redundant Model Selection for Ensemble Construction

Sorts models by AUC on the chosen validation set (best first), then applies a pairwise **descriptor-overlap filter**: a model is kept only if none of its descriptor pairs already appear in any previously accepted model. This ensures chemical diversity in the final ensemble.

> **Note:** `BEST_FEATURES_FILE_S3` must match the filename produced by Step 1 (`best_features_<MODEL_TYPE>_<TARGET_NAME>.csv`) and `INDIVIDUAL_RESULTS_FILE_S3` must match the filename produced by Step 2 (`individual_results_<MODEL_TYPE>_<TARGET_NAME>.csv`).

**Outputs:** ranked model list (CSV), selected model list (CSV), model names (TXT).


In [ ]:
print("=" * 70)
print("STEP 3 — Non-redundant model selection")
print("=" * 70)

# ----- USER CONFIGURATION ----------------------------------------------------
WORKING_DIR_S3           = Path("/Users/francocaram/Library/CloudStorage/GoogleDrive-francocaram8@gmail.com/My Drive/DOCTORADO/Cdifficile_ML/Data_sets")
BEST_FEATURES_FILE_S3    = "best_features_OLS_CDIFF.csv"
INDIVIDUAL_RESULTS_FILE_S3 = "individual_results_OLS_CDIFF.csv"
RANKING_METRIC_S3        = "AUC_val2"   # Column used to rank models (e.g., "AUC_val1")
# ----- END CONFIGURATION -----------------------------------------------------

directorio_s3 = str(WORKING_DIR_S3)

features_df_s3  = pd.read_csv(directorio_s3 + "/" + BEST_FEATURES_FILE_S3,
                               sep=",", index_col="Unnamed: 0")
results_df_s3   = pd.read_csv(directorio_s3 + "/" + INDIVIDUAL_RESULTS_FILE_S3,
                               sep=",", index_col="MODEL")

# Merge and sort by chosen metric
merged_s3  = pd.concat([results_df_s3[[RANKING_METRIC_S3]], features_df_s3], axis=1)
sorted_s3  = merged_s3.sort_values(by=[RANKING_METRIC_S3], ascending=False)
sorted_s3.to_csv(directorio_s3 + "/" + "models_ranked.csv", sep=",")

# Build descriptor-pair lists for each model
pair_lists_s3 = []
model_ids_s3  = []
for index, row in sorted_s3.iterrows():
    descs = [x for x in list(row[1:]) if str(x) != "nan"]
    pairs = [[d1, d2] for d1 in descs for d2 in descs if d1 != d2]
    pair_lists_s3.append(pairs)
    model_ids_s3.append(index)

# Greedy non-redundancy filter
accumulated_pairs_s3 = pair_lists_s3[0][:]   # Seed with pairs of the top-ranked model
selected_model_ids_s3 = []

for i, model_pairs in enumerate(pair_lists_s3):
    novel_pairs = [p for p in model_pairs if p not in accumulated_pairs_s3]
    if len(model_pairs) == len(novel_pairs):    # All pairs are new → accept model
        selected_model_ids_s3.append(model_ids_s3[i])
        accumulated_pairs_s3.extend(model_pairs)

# Save selected model names
with open(directorio_s3 + "/" + "models_to_combine.txt", "w") as f:
    for name in selected_model_ids_s3:
        f.write(name + "\n")

# Save selected model descriptor table
selected_rows_s3 = []
for index, row in sorted_s3.iterrows():
    if index in selected_model_ids_s3:
        selected_rows_s3.append(list(row[0:]))

selected_df_s3 = pd.DataFrame(selected_rows_s3)
selected_df_s3["model_name"] = selected_model_ids_s3
selected_df_s3.to_csv(directorio_s3 + "/" + "selected_models.csv", sep=",")

print("Step 3 complete.\n")


---

## Step 4 — Ensemble Generation and AUC/AUPR Evaluation

Combines individual model scores using five aggregation strategies:

| Strategy | Description |
|----------|-------------|
| **MIN** | Minimum score across models |
| **MEAN** | Arithmetic mean of scores |
| **PROD** | Product of scores |
| **RANK** | Mean of rank-normalised scores |
| **VOTING** | Weighted voting scheme |

For each strategy and number of combined models (1 to `NUM_MODELS_TO_COMBINE`), AUC and AUPR are estimated by **stratified bootstrap sampling**.

**Outputs:** score tables (CSV) and metrics-vs-N-models tables (CSV).


In [ ]:
print("=" * 70)
print("STEP 4 — Ensemble generation and evaluation")
print("=" * 70)

# ----- USER CONFIGURATION ----------------------------------------------------
TARGET_NAME_S4          = "CDIFF"        # Must match TARGET_NAME_S2 exactly
MODEL_TYPE_S4           = "OLS"           # Must match MODEL_TYPE_S2 exactly
WORKING_DIR_S4          = Path("/Users/francocaram/Library/CloudStorage/GoogleDrive-francocaram8@gmail.com/My Drive/DOCTORADO/Cdifficile_ML/Data_sets")
RANK_BY_VALIDATION_S4   = "val2"           # Test set used to order models
VALIDATION_SETS_S4      = ["val1", "val2"] # All sets to be evaluated
NUM_MODELS_TO_COMBINE_S4 = 100             # Maximum number of models in the ensemble
BOOTSTRAP_REPS_S4       = 100             # Bootstrap repetitions per ensemble size
BOOTSTRAP_FRACTION_S4   = 0.75            # Fraction of compounds sampled per repetition
# ----- END CONFIGURATION -----------------------------------------------------

from sklearn.model_selection import train_test_split

directorio_s4 = str(WORKING_DIR_S4)

# Load selected model list
models_to_combine_s4 = []
with open(directorio_s4 + "/" + "models_to_combine.txt", "r") as f:
    models_to_combine_s4 = [line.strip() for line in f]

# Load individual model results and sort by chosen validation AUC
ind_results_s4 = pd.read_csv(
    directorio_s4 + "/" + "individual_results_" + MODEL_TYPE_S4 + "_" + TARGET_NAME_S4 + ".csv",  # Must match MODEL_TYPE_S2 and TARGET_NAME_S2 from Step 2
    index_col=None
)
ind_results_s4.sort_values("AUC_" + RANK_BY_VALIDATION_S4, ascending=False, inplace=True)
model_order_s4 = models_to_combine_s4 if models_to_combine_s4 \
    else ind_results_s4["MODEL"].values.tolist()

for base in VALIDATION_SETS_S4:
    # Load individual scores
    scores_s4 = pd.read_csv(
        directorio_s4 + "/" + "individual_scores_" + base + "_" + MODEL_TYPE_S4 + "_" + TARGET_NAME_S4 + ".csv",
        index_col=0
    )
    y_true_s4 = scores_s4["clase"]

    # Pre-compute ranked and voting versions of scores
    scores_rank_s4   = scores_s4.copy()
    scores_voting_s4 = scores_s4.copy()
    for mod in model_order_s4:
        scores_rank_s4[mod]   = scores_rank_s4[mod].rank()
        scores_voting_s4[mod] = scores_voting_s4[mod].rank(ascending=False)

    n_compounds_s4 = len(y_true_s4)
    voting_matrix_s4 = scores_voting_s4[model_order_s4].apply(
        lambda x: (11 - (x / (n_compounds_s4 * 0.02))).astype(int)
    )
    voting_matrix_s4[voting_matrix_s4 < 0] = 0
    voting_df_s4 = pd.concat([y_true_s4, voting_matrix_s4], axis=1)

    # Initialize result containers
    (n_models_list,
     auc_min, sd_min, aupr_min, sdaupr_min, sc_min, hdr_min,
     auc_mean, sd_mean, aupr_mean, sdaupr_mean, sc_mean, hdr_mean,
     auc_prod, sd_prod, aupr_prod, sdaupr_prod, sc_prod, hdr_prod,
     auc_rank, sd_rank, aupr_rank, sdaupr_rank, sc_rank, hdr_rank,
     auc_vote, sd_vote, aupr_vote, sdaupr_vote, sc_vote, hdr_vote
     ) = ([], [], [], [], [], [], ["clase"],
           [], [], [], [], [], ["clase"],
           [], [], [], [], [], ["clase"],
           [], [], [], [], [], ["clase"],
           [], [], [], [], [], ["clase"])

    combo_idx = 1
    for n in range(1, NUM_MODELS_TO_COMBINE_S4 + 1):
        # Compute ensemble scores for current n models
        subset_scores_s4 = scores_s4[model_order_s4[:n]]
        y_min_s4  = subset_scores_s4.min(1)
        y_mean_s4 = subset_scores_s4.mean(1)
        y_prod_s4 = subset_scores_s4.product(1)
        y_rank_s4 = (scores_rank_s4[model_order_s4[:n]] / n_compounds_s4).mean(1)
        y_vote_s4 = voting_df_s4[model_order_s4[:n]].mean(1)

        # Bootstrap evaluation for each aggregation strategy
        for (y_ens, auc_l, sd_l, aupr_l, sdaupr_l, sc_l, hdr_l, label) in [
            (y_min_s4,  auc_min,  sd_min,  aupr_min,  sdaupr_min,  sc_min,  hdr_min,  "MIN"),
            (y_mean_s4, auc_mean, sd_mean, aupr_mean, sdaupr_mean, sc_mean, hdr_mean, "MEAN"),
            (y_prod_s4, auc_prod, sd_prod, aupr_prod, sdaupr_prod, sc_prod, hdr_prod, "PROD"),
            (y_rank_s4, auc_rank, sd_rank, aupr_rank, sdaupr_rank, sc_rank, hdr_rank, "RANK"),
            (y_vote_s4, auc_vote, sd_vote, aupr_vote, sdaupr_vote, sc_vote, hdr_vote, "VOTE"),
        ]:
            aucs_b, auprs_b = [], []
            for rep in range(1, BOOTSTRAP_REPS_S4 + 1):
                cl_tr, _, sc_tr, _ = train_test_split(
                    y_true_s4, y_ens,
                    test_size=1 - BOOTSTRAP_FRACTION_S4,
                    random_state=rep, stratify=y_true_s4
                )
                aucs_b.append(roc_auc_score(cl_tr, sc_tr))
                auprs_b.append(average_precision_score(cl_tr, sc_tr))
            auc_l.append(statistics.mean(aucs_b))
            sd_l.append(statistics.stdev(aucs_b))
            aupr_l.append(statistics.mean(auprs_b))
            sdaupr_l.append(statistics.stdev(auprs_b))
            sc_l.append(y_ens)
            hdr_l.append(label + "_" + str(combo_idx))

        n_models_list.append(combo_idx)
        combo_idx += 1

    # Save score tables
    for sc_list, headers, label in [
        (sc_min,  hdr_min,  "MIN"),  (sc_mean, hdr_mean, "MEAN"),
        (sc_prod, hdr_prod, "PROD"), (sc_rank, hdr_rank, "RANK"),
        (sc_vote, hdr_vote, "VOTE"),
    ]:
        df_sc = pd.DataFrame(sc_list).T
        pd.concat([y_true_s4, df_sc], axis=1).to_csv(
            directorio_s4 + "/" + "scores_" + label + "_" + base + "_" + MODEL_TYPE_S4 + "_" + TARGET_NAME_S4 + ".csv",
            sep=",", index=True, header=headers
        )

    # Save metrics table
    metrics_s4 = pd.DataFrame(list(zip(
        n_models_list,
        auc_min, sd_min, auc_mean, sd_mean, auc_prod, sd_prod, auc_rank, sd_rank, auc_vote, sd_vote,
        aupr_min, sdaupr_min, aupr_mean, sdaupr_mean, aupr_prod, sdaupr_prod,
        aupr_rank, sdaupr_rank, aupr_vote, sdaupr_vote,
    )), columns=[
        "Num_models",
        "AUC_MIN", "sd_MIN", "AUC_MEAN", "sd_MEAN",
        "AUC_PROD", "sd_PROD", "AUC_RANK", "sd_RANK", "AUC_VOTE", "sd_VOTE",
        "AUPR_MIN", "sd_AUPR_MIN", "AUPR_MEAN", "sd_AUPR_MEAN",
        "AUPR_PROD", "sd_AUPR_PROD", "AUPR_RANK", "sd_AUPR_RANK", "AUPR_VOTE", "sd_AUPR_VOTE",
    ])
    metrics_s4.to_csv(
        directorio_s4 + "/" + "ensemble_metrics_" + base + "_" + MODEL_TYPE_S4 + "_" + TARGET_NAME_S4 + ".csv",
        sep=",", index=False
    )

print("Step 4 complete.\n")


---

## Step 5 — Interactive AUC-vs-Number-of-Models Curves

Reads the metrics table produced in Step 4 and generates an interactive **Plotly** figure showing AUC-ROC and AUPR as a function of the number of ensembled models, with error bars (standard deviation). Each aggregation strategy is displayed in a distinct colour.

**Outputs:** interactive HTML figure with a configurable image-download button.


In [ ]:
print("=" * 70)
print("STEP 5 — AUC-vs-number-of-models curves")
print("=" * 70)

# ----- USER CONFIGURATION ----------------------------------------------------
TARGET_NAME_S5  = "CDIFF"
MODEL_TYPE_S5   = "OLS"
VALIDATION_S5   = "val1"                              # Which test set to plot
WORKING_DIR_S5  = Path("/Users/francocaram/Library/CloudStorage/GoogleDrive-francocaram8@gmail.com/My Drive/DOCTORADO/Cdifficile_ML/Data_sets")
IMAGE_FORMAT_S5 = "png"    # Download format: png, svg, jpeg, webp
IMAGE_HEIGHT_S5 = 800      
IMAGE_WIDTH_S5  = 1600     
IMAGE_SCALE_S5  = 10       # Scale factor for font/axis sizes in exported image
# ----- END CONFIGURATION -----------------------------------------------------

import plotly
import plotly.graph_objects as go

directorio_s5 = str(WORKING_DIR_S5)

metrics_s5   = pd.read_csv(
    directorio_s5 + "/" + "ensemble_metrics_" + VALIDATION_S5 + "_" + MODEL_TYPE_S5 + "_" + TARGET_NAME_S5 + ".csv"
)
n_models_s5  = metrics_s5["Num_models"]

strategy_colors_s5 = {
    "MIN":  "rgba(255, 0, 0, 1)",
    "MEAN": "rgba(47, 10, 255, 1)",
    "PROD": "rgba(0, 200, 0, 1)",
    "RANK": "rgba(200, 180, 0, 1)",
    "VOTE": "rgba(140, 140, 140, 1)",
}

fig_s5 = go.Figure()
for strategy, color in strategy_colors_s5.items():
    for metric, sd_prefix in [
        ("AUC",   "sd_"),
        ("AUPR",  "sd_AUPR_"),
    ]:
        col_val = metric + "_" + strategy
        col_sd  = sd_prefix + strategy
        fig_s5.add_trace(go.Scatter(
            x=n_models_s5, y=metrics_s5[col_val],
            mode="lines+markers",
            name=metric + " " + strategy,
            marker_color=color,
            error_y=dict(type="data", symmetric=True, array=metrics_s5[col_sd])
        ))

fig_s5.update_layout(
    xaxis_title="Number of ensembled models",
    yaxis_title="Metric value",
    legend_title=None,
    font=dict(family="Courier New, monospace", size=24, color="black")
)
fig_s5.show()

config_s5 = {
    "toImageButtonOptions": {
        "format": IMAGE_FORMAT_S5,
        "filename": "metrics_vs_N_models_" + VALIDATION_S5 + "_" + TARGET_NAME_S5,
        "height": IMAGE_HEIGHT_S5, "width": IMAGE_WIDTH_S5, "scale": IMAGE_SCALE_S5
    }
}
plotly.offline.plot(
    fig_s5,
    filename=directorio_s5 + "/" + "metrics_vs_N_models_" + VALIDATION_S5 + "_" + TARGET_NAME_S5 + ".html",
    config=config_s5
)

print("Step 5 complete.\n")


---

## Step 6 — Internal Validations (Fisher Y-Randomization and Leave-Group-Out CV)

Performs two types of internal validation on a user-defined list of models:

- **Fisher y-randomization:** the response variable (class) is randomly shuffled multiple times and a model is re-fitted each time. If the original model's correctly classified fraction is substantially above the randomised mean, the model captures real structure-activity information.
- **Leave-Group-Out (LGO) cross-validation:** the training set is repeatedly split into a training partition and a held-out test group (stratified). Descriptor selection is re-run on each training partition to avoid data leakage. The mean and SD of the correctly classified fraction on the held-out group summarise predictive performance.

**Outputs:** internal validation results (XLSX) and a parameter log (TXT).


In [ ]:
print("=" * 70)
print("STEP 6 — Internal validations (Fisher + LGO)")
print("=" * 70)

# ----- USER CONFIGURATION ----------------------------------------------------
WORKING_DIR_EX        = Path(r"/Users/francocaram/Library/CloudStorage/GoogleDrive-francocaram8@gmail.com/My Drive/DOCTORADO/Cdifficile_ML/Data_sets")
TARGET_NAME_EX        = "CDIFF_SINTC"
TRAINING_FILE_EX      = "training_set.csv"
MODELS_TO_VALIDATE_EX = [6727, 6551, 2262, 8241, 8646]  # Model indices to validate

FISHER_REPS_EX        = 20    # Number of y-randomization repetitions
LGO_REPS_EX           = 20    # Number of LGO cross-validation repetitions
LGO_TEST_FRACTION_EX  = 0.1   # Fraction of compounds held out per LGO repetition

# Descriptor generation parameters — must match those used in Step 1
VARIANCE_THRESHOLD_EX  = 0.05
NUM_SUBSETS_EX         = 10000
DESCRIPTORS_PER_SUBSET_EX = 200
CORRELATION_METHOD_EX  = "pearson"
CORRELATION_LIMIT_EX   = 0.5
SIGNIFICANCE_LEVEL_EX  = 0.05
MAX_DESCRIPTORS_EX     = 6
# ----- END CONFIGURATION -----------------------------------------------------

directorio_ex = str(WORKING_DIR_EX)

# Load and clean training data
raw_ex = pd.read_csv(directorio_ex + "/" + TRAINING_FILE_EX, sep=",", header="infer", index_col=None)
data_ex = raw_ex.dropna(axis=1)

cols_ex       = list(data_ex)
nc_ex         = data_ex[["NAME", "clase"]]
desc_names_ex = cols_ex[2:]
descs_ex      = data_ex[desc_names_ex]

# Variance filter
vs_ex = VarianceThreshold(VARIANCE_THRESHOLD_EX)
vs_ex.fit(descs_ex)
descs_ok_ex = descs_ex[descs_ex.columns[vs_ex.get_support(indices=True)]]

# Rebuild subsets (same random seed sequence as Step 1)
subsets_ex = []
idx_ex = 0
while idx_ex < NUM_SUBSETS_EX:
    subset = descs_ok_ex.sample(DESCRIPTORS_PER_SUBSET_EX, axis=1, random_state=idx_ex)
    corr   = subset.corr(CORRELATION_METHOD_EX).abs()
    upper  = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
    drop   = [c for c in upper.columns if any(upper[c] > CORRELATION_LIMIT_EX)]
    subsets_ex.append(pd.concat([nc_ex, subset.drop(drop, axis=1)], axis=1))
    idx_ex += 1

def forward_stepwise_ex(X, y, sig=0.05, max_f=5):
    """Forward stepwise selection — internal validation version."""
    all_f = X.columns.tolist()
    while len(all_f) > 0:
        remaining = list(set(all_f) - set(sel_feats_ex))
        pv = pd.Series(index=remaining, dtype=float)
        for feat in remaining:
            ols = sm.OLS(y, sm.add_constant(X[sel_feats_ex + [feat]])).fit()
            pv[feat] = ols.pvalues[feat]
        if pv.min() < sig and len(sel_feats_ex) < max_f:
            sel_feats_ex.append(pv.idxmin())
        else:
            break

# --- Original model performance ---
model_labels_ex   = []
acc_original_ex   = []

for midx in MODELS_TO_VALIDATE_EX:
    label = "MODEL_" + str(midx)
    model_labels_ex.append(label)
    sel_feats_ex = []
    sub  = subsets_ex[midx - 1]
    X_s  = sub[list(sub)[2:]].reset_index(drop=True)
    y_s  = sub["clase"].reset_index(drop=True)
    forward_stepwise_ex(X_s, y_s, SIGNIFICANCE_LEVEL_EX, MAX_DESCRIPTORS_EX)
    X_f  = sub[sel_feats_ex]
    reg  = LinearRegression(fit_intercept=True, copy_X=True, n_jobs=None)
    reg.fit(X_f, y_s)
    pred = pd.Series([1 if v > 0.5 else 0 for v in reg.predict(X_f)])
    acc_original_ex.append(sum(y_s.eq(pred)) / len(y_s))

# --- Fisher y-randomization ---
mean_acc_fisher_ex = []
sd_acc_fisher_ex   = []

for midx in MODELS_TO_VALIDATE_EX:
    sel_feats_ex = []
    sub  = subsets_ex[midx - 1]
    X_s  = sub[list(sub)[2:]].reset_index(drop=True)
    y_s  = sub["clase"].reset_index(drop=True)
    acc_rand = []
    for rep in range(FISHER_REPS_EX + 1):
        sel_feats_ex = []
        y_rand = sub["clase"].sample(frac=1, random_state=rep).reset_index(drop=True)
        forward_stepwise_ex(X_s, y_rand, SIGNIFICANCE_LEVEL_EX, MAX_DESCRIPTORS_EX)
        if not sel_feats_ex:
            continue
        X_f = sub[sel_feats_ex]
        reg = LinearRegression(fit_intercept=True, copy_X=True, n_jobs=None)
        reg.fit(X_f, y_rand)
        pred_rand = pd.Series([1 if v > 0.5 else 0 for v in reg.predict(X_f)])
        acc_rand.append(sum(y_s.eq(pred_rand)) / len(y_s))
    mean_acc_fisher_ex.append(statistics.mean(acc_rand))
    sd_acc_fisher_ex.append(statistics.stdev(acc_rand))

# --- Leave-Group-Out cross-validation ---
mean_acc_lgo_ex = []
sd_acc_lgo_ex   = []

for midx in MODELS_TO_VALIDATE_EX:
    sub  = subsets_ex[midx - 1]
    X_s  = sub[list(sub)[2:]].reset_index(drop=True)
    y_s  = sub["clase"].reset_index(drop=True)
    acc_lgo = []
    for rep in range(LGO_REPS_EX + 1):
        sel_feats_ex = []
        X_tr, X_te, y_tr, y_te = train_test_split(
            X_s, y_s, test_size=LGO_TEST_FRACTION_EX, random_state=rep, stratify=y_s
        )
        forward_stepwise_ex(X_tr, y_tr, SIGNIFICANCE_LEVEL_EX, MAX_DESCRIPTORS_EX)
        reg = LinearRegression(fit_intercept=True, copy_X=True, n_jobs=None)
        reg.fit(X_tr[sel_feats_ex], y_tr)
        pred_te = pd.Series([1 if v > 0.5 else 0 for v in reg.predict(X_te[sel_feats_ex])])
        y_te_s  = pd.Series(y_te.tolist())
        acc_lgo.append(sum(y_te_s.eq(pred_te)) / len(y_te_s))
    mean_acc_lgo_ex.append(statistics.mean(acc_lgo))
    sd_acc_lgo_ex.append(statistics.stdev(acc_lgo))

# Save results
val_df_ex = pd.DataFrame({
    "Model":               model_labels_ex,
    "Acc_original":        acc_original_ex,
    "Mean_Acc_Fisher":     mean_acc_fisher_ex,
    "SD_Acc_Fisher":       sd_acc_fisher_ex,
    "Mean_Acc_LGO":        mean_acc_lgo_ex,
    "SD_Acc_LGO":          sd_acc_lgo_ex,
})
val_df_ex.to_excel(
    directorio_ex + "/" + "internal_validations_" + TARGET_NAME_EX + ".xlsx",
    header=True, index=False
)

today_ex = date.today().strftime("%d/%m/%Y")
with open(directorio_ex + "/" + "internal_validations_info.txt", "w") as log:
    log.write("Internal validation log — " + today_ex + "\n\n")
    log.write("Note: descriptor selection was re-run at each repetition.\n\n")
    log.write("Fisher y-randomization\n")
    log.write("  Repetitions     : " + str(FISHER_REPS_EX) + "\n\n")
    log.write("Leave-Group-Out cross-validation\n")
    log.write("  Test fraction   : " + str(LGO_TEST_FRACTION_EX) + "\n")
    log.write("  Repetitions     : " + str(LGO_REPS_EX) + "\n")

print("Step 6 complete.\n")


---

## Step 7 — Detailed Evaluation of a Selected Ensemble

Evaluates a single chosen ensemble (strategy + number of models) on one or more validation sets. Computes AUC-ROC, RIE, Enrichment Factor (EF), and AUPR with their standard deviations via **stratified bootstrap sampling**. Saves ROC and Precision-Recall curves as interactive HTML files.

**Outputs:** summary metrics table (CSV) and ROC/PR curve HTML files per validation set.


In [ ]:
print("=" * 70)
print("STEP 7 — Detailed ensemble evaluation")
print("=" * 70)

# ----- USER CONFIGURATION ----------------------------------------------------
TARGET_NAME_S6   = "CDIFF"
MODEL_TYPE_S6    = "OLS"
WORKING_DIR_S6   = Path("/Users/francocaram/Library/CloudStorage/GoogleDrive-francocaram8@gmail.com/My Drive/DOCTORADO/Cdifficile_ML/Data_sets")
VALIDATION_SETS_S6 = ["val1", "val2"]
ENSEMBLE_S6      = "MIN"    # Aggregation strategy: MIN, MEAN, PROD, RANK, VOTE
N_MODELS_S6      = "5"      # Number of models in the chosen ensemble (as string)
BOOTSTRAP_REPS_S6     = 100
BOOTSTRAP_FRACTION_S6 = 0.75
RIE_ALPHA_S6     = 20
IMAGE_FORMAT_S6  = "svg"
IMAGE_HEIGHT_S6  = 800
IMAGE_WIDTH_S6   = 1600
IMAGE_SCALE_S6   = 1
# ----- END CONFIGURATION -----------------------------------------------------

import math
from rdkit.ML.Scoring import Scoring
import plotly as py
from plotly.subplots import make_subplots
from sklearn.metrics import roc_curve

directorio_s6   = str(WORKING_DIR_S6)
summary_s6      = []

for base in VALIDATION_SETS_S6:
    scores_df_s6 = pd.read_csv(
        directorio_s6 + "/" + "scores_" + ENSEMBLE_S6 + "_" + base
        + "_" + MODEL_TYPE_S6 + "_" + TARGET_NAME_S6 + ".csv"
    )
    ens_col_s6 = ENSEMBLE_S6 + "_" + N_MODELS_S6
    scores_df_s6.sort_values(ens_col_s6, ascending=False, inplace=True)
    y_ens_s6  = list(scores_df_s6[ens_col_s6])
    y_true_s6 = list(scores_df_s6["clase"])

    aucs_s6, ries_s6, efs_s6, auprs_s6 = [], [], [], []

    for rep in range(0, BOOTSTRAP_REPS_S6 + 1):
        sc_tr, _, cl_tr, _ = train_test_split(
            y_ens_s6, y_true_s6,
            test_size=1 - BOOTSTRAP_FRACTION_S6,
            random_state=rep, stratify=y_true_s6
        )
        arr_s6 = pd.DataFrame(list(zip(sc_tr, cl_tr))).sort_values(0, ascending=False)
        tup_s6 = tuple(zip(arr_s6[0], arr_s6[1]))

        aucs_s6.append(Scoring.CalcAUC(scores=tup_s6, col=1))
        ries_s6.append(Scoring.CalcRIE(scores=tup_s6, col=1, alpha=RIE_ALPHA_S6))
        efs_s6.append(Scoring.CalcEnrichment(scores=tup_s6, col=1, fractions=[0.01])[0])
        auprs_s6.append(average_precision_score(cl_tr, sc_tr))

    n_total_s6   = len(y_true_s6)
    n_actives_s6 = sum(y_true_s6)
    n_ef_cutoff  = math.ceil(n_total_s6 * 0.01)
    efmax_s6     = (n_actives_s6 / n_ef_cutoff) / (n_actives_s6 / n_total_s6) \
                    if n_ef_cutoff > n_actives_s6 else 1.0 / (n_actives_s6 / n_total_s6)
    rie_max_s6   = Scoring.CalcRIE(
        scores=tuple(zip(y_ens_s6, y_true_s6)), col=1, alpha=RIE_ALPHA_S6
    )

    summary_s6.append([
        base,
        statistics.mean(aucs_s6), statistics.stdev(aucs_s6),
        statistics.mean(ries_s6), statistics.stdev(ries_s6), rie_max_s6,
        statistics.mean(efs_s6), statistics.stdev(efs_s6), efmax_s6,
        statistics.mean(auprs_s6), statistics.stdev(auprs_s6), "1"
    ])

    # ROC and PR curves
    fpr_s6, tpr_s6, _ = roc_curve(y_true=y_true_s6, y_score=y_ens_s6)
    prec_s6, rec_s6, _ = precision_recall_curve(y_true=y_true_s6, y_score=y_ens_s6)

    title_roc_s6 = (
        "ROC curve — " + base
        + "  (AUC = " + format(statistics.mean(aucs_s6), ".4f")
        + " ± " + format(statistics.stdev(aucs_s6), ".4f") + ")"
    )
    title_pr_s6 = (
        "PR curve — " + base
        + "  (AUPR = " + format(statistics.mean(auprs_s6), ".4f")
        + " ± " + format(statistics.stdev(auprs_s6), ".4f") + ")"
    )

    fig_s6 = make_subplots(rows=1, cols=2, subplot_titles=(title_roc_s6, title_pr_s6))
    fig_s6.add_trace(go.Scatter(x=fpr_s6, y=tpr_s6, fill="tozeroy", showlegend=False), row=1, col=1)
    fig_s6.add_shape(type="line", line=dict(dash="dash"), x0=0, x1=1, y0=0, y1=1, row=1, col=1)
    fig_s6.add_trace(go.Scatter(x=rec_s6, y=prec_s6, fill="tozeroy", showlegend=False), row=1, col=2)
    fig_s6.add_shape(type="line", line=dict(dash="dash"), x0=0, x1=1, y0=1, y1=0, row=1, col=2)

    py.offline.plot(
        fig_s6,
        filename=directorio_s6 + "/" + "ROC_PR_" + ENSEMBLE_S6 + "_" + N_MODELS_S6
                 + "_" + TARGET_NAME_S6 + "_" + base + ".html",
        config={"toImageButtonOptions": {
            "format": IMAGE_FORMAT_S6, "filename": "ROC_PR_" + base,
            "height": IMAGE_HEIGHT_S6, "width": IMAGE_WIDTH_S6, "scale": IMAGE_SCALE_S6
        }}
    )

final_s6 = pd.DataFrame(summary_s6).rename(columns={
    0: "set", 1: "AUC", 2: "sd_AUC",
    3: "RIE_" + str(RIE_ALPHA_S6), 4: "sd_RIE", 5: "RIEmax_" + str(RIE_ALPHA_S6),
    6: "EF(1%)", 7: "sd_EF(1%)", 8: "EFmax(1%)",
    9: "AUPR", 10: "sd_AUPR", 11: "AUPRmax"
})
final_s6.to_csv(
    directorio_s6 + "/" + "ensemble_evaluation_" + ENSEMBLE_S6 + "_" + N_MODELS_S6
    + "_" + TARGET_NAME_S6 + ".csv",
    sep=",", index=False
)

print("Step 7 complete.\n")


---

## Step 8 — Positive Predictive Value (PPV) Surface

Computes the **PPV** as a function of disease prevalence (Ya) and the ratio sensitivity/specificity (Se/Sp) for a chosen ensemble and cutoff. The result is visualised as an interactive 3D surface using Plotly. A cutoff table with Se, Sp, Se/Sp, and PPV at two reference prevalence values is also saved. This decision will be necessary in step 12.

**Outputs:** interactive 3D PPV surface (HTML) and cutoff table (CSV).


In [ ]:
print("=" * 70)
print("STEP 8 — PPV surface plot")
print("=" * 70)

# ----- USER CONFIGURATION ----------------------------------------------------
TARGET_NAME_S7  = "CDIFF"
MODEL_TYPE_S7   = "OLS"
VALIDATION_S7   = "val2"
ENSEMBLE_S7     = "MIN"
N_MODELS_S7     = "5"
WORKING_DIR_S7  = Path("/Users/francocaram/Library/CloudStorage/GoogleDrive-francocaram8@gmail.com/My Drive/DOCTORADO/Cdifficile_ML/Data_sets")
REF_PREVALENCES_S7 = [0.001, 0.01]  # Reference Ya values for the cutoff table
IMAGE_FORMAT_S7 = "svg"
IMAGE_HEIGHT_S7 = 800
IMAGE_WIDTH_S7  = 1600
IMAGE_SCALE_S7  = 1
# ----- END CONFIGURATION -----------------------------------------------------

directorio_s7 = str(WORKING_DIR_S7)

scores_s7  = pd.read_csv(
    directorio_s7 + "/" + "scores_" + ENSEMBLE_S7 + "_" + VALIDATION_S7
    + "_" + MODEL_TYPE_S7 + "_" + TARGET_NAME_S7 + ".csv"
)
y_ens_s7   = scores_s7[ENSEMBLE_S7 + "_" + N_MODELS_S7]
y_true_s7  = scores_s7["clase"]

fpr_s7, tpr_s7, thresholds_s7 = roc_curve(y_score=y_ens_s7, y_true=y_true_s7,
                                            drop_intermediate=False)
thresholds_s7 = thresholds_s7[1:]
se_s7     = tpr_s7[1:]
sp_s7     = (1 - fpr_s7)[1:]
se_sp_s7  = se_s7 / sp_s7
idx_range = list(range(len(se_s7)))

# Cutoff table with PPV at reference prevalences
ppv_ref_s7 = []
for prev in REF_PREVALENCES_S7:
    ppv_ref_s7.append([(se_s7[i] * prev) / (se_s7[i] * prev + (1 - sp_s7[i]) * (1 - prev))
                       for i in idx_range])

cutoff_df_s7 = pd.DataFrame({
    "Cutoff": thresholds_s7,
    "Se": se_s7,
    "Sp": sp_s7,
    "Se/Sp": se_sp_s7,
    "PPV (Ya=0.001)": ppv_ref_s7[0],
    "PPV (Ya=0.01)":  ppv_ref_s7[1]
})
cutoff_df_s7.to_csv(
    directorio_s7 + "/" + "cutoffs_" + ENSEMBLE_S7 + "_" + N_MODELS_S7
    + "_" + VALIDATION_S7 + ".csv",
    sep=",", index=False
)

# PPV surface over a prevalence grid
ya_values_s7 = np.arange(1e-11, 0.012, 0.001).tolist()
ppv_surface_s7 = pd.DataFrame(
    [[( se_s7[i] * ya) / (se_s7[i] * ya + (1 - sp_s7[i]) * (1 - ya))
      for i in idx_range]
     for ya in ya_values_s7]
).T

fig_s7 = go.Figure(go.Surface(
    x=ya_values_s7, y=se_sp_s7, z=ppv_surface_s7,
    opacity=1,
    contours={
        "x": {"show": False, "color": "white"},
        "y": {"show": False, "color": "white"},
        "z": {"show": False, "color": "white"},
    }
))
fig_s7.update_layout(
    scene=dict(
        xaxis=dict(title="Ya (prevalence)", nticks=10, range=[0, 0.011],
                   backgroundcolor="rgb(200,200,230)", gridcolor="white",
                   showbackground=True, zerolinecolor="white"),
        yaxis=dict(title="Se/Sp", nticks=10, range=[0, 2],
                   backgroundcolor="rgb(230,200,230)", gridcolor="white",
                   showbackground=True, zerolinecolor="white"),
        zaxis=dict(title="PPV", nticks=10, range=[0, 1],
                   backgroundcolor="rgb(230,230,200)", gridcolor="white",
                   showbackground=True, zerolinecolor="white"),
    ),
    margin=dict(r=20, l=10, b=10, t=10),
    font=dict(size=16, color="black")
)
fig_s7.show()

plotly.offline.plot(
    fig_s7,
    filename=directorio_s7 + "/" + "PPV_" + ENSEMBLE_S7 + "_" + N_MODELS_S7
             + "_" + TARGET_NAME_S7 + "_" + VALIDATION_S7 + ".html",
    config={"toImageButtonOptions": {
        "format": IMAGE_FORMAT_S7,
        "filename": "PPV_" + ENSEMBLE_S7 + "_" + N_MODELS_S7,
        "height": IMAGE_HEIGHT_S7, "width": IMAGE_WIDTH_S7, "scale": IMAGE_SCALE_S7
    }}
)

print("Step 8 complete.\n")


---

## Step 9 — Prospective Virtual Screening on an External Compound Library

Applies all models of the ensemble to an external chemical library (no class labels required). For each compound, individual model scores are computed and then aggregated using the chosen strategy (MIN or MEAN). Compounds with any missing descriptor value are excluded from scoring and reported separately.

**Outputs:** descriptor value table (CSV) and ensemble score table (CSV).


In [ ]:
print("=" * 70)
print("STEP 9 — Prospective virtual screening")
print("=" * 70)

# ----- USER CONFIGURATION ----------------------------------------------------
WORKING_DIR_S8         = Path("/Users/francocaram/Library/CloudStorage/GoogleDrive-francocaram8@gmail.com/My Drive/DOCTORADO/Cdifficile_ML/Data_sets")
TARGET_NAME_S8         = "CDIFF"
MODEL_TYPE_S8          = "OLS"
ENSEMBLE_STRATEGY_S8   = ["MIN"]       # One strategy at a time: MIN, MEAN, PROD, RANK, or VOTE
N_MODELS_SCREEN_S8     = 5
SCREENING_LIB_NAME_S8  = "DrugBank"    # Label for output filenames
TRAINING_FILE_S8       = "training_set.csv"
SCREENING_LIB_FILE_S8  = "descr_drugbank.txt"  # Tab-separated
BEST_FEATURES_FILE_S8  = "selected_models.csv"        # Output from Step 3
# ----- END CONFIGURATION -----------------------------------------------------

directorio_s8 = str(WORKING_DIR_S8)

# Load training data
training_s8 = pd.read_csv(directorio_s8 + "/" + TRAINING_FILE_S8, sep=",",
                           header="infer", index_col=None)
training_s8 = training_s8.dropna(axis=1)

# Parse best-features file
selected_df_s8 = pd.read_csv(directorio_s8 + "/" + BEST_FEATURES_FILE_S8)
desc_cols_s8   = [c for c in selected_df_s8.columns if c not in ["Unnamed: 0", "0", "model_name"]]
best_feats_s8, model_names_s8, all_descs_s8 = [], [], []
for _, row in selected_df_s8.iterrows():
    feats = list(filter(lambda x: isinstance(x, str) and x.strip() != "", row[desc_cols_s8]))
    best_feats_s8.append(feats)
    model_names_s8.append(row["model_name"])
    for desc in feats:
        if desc not in all_descs_s8:
            all_descs_s8.append(desc)

# Load screening library, subset to required descriptors, drop rows with NaN
lib_raw_s8 = pd.read_csv(directorio_s8 + "/" + SCREENING_LIB_FILE_S8, sep="\t", index_col="NAME")
lib_subset_s8 = lib_raw_s8[all_descs_s8]
lib_clean_s8  = lib_subset_s8.dropna(axis=0, how="any")
lib_clean_s8.to_csv(
    directorio_s8 + "/" + "descriptor_values_" + ENSEMBLE_STRATEGY_S8[0]
    + "_" + str(N_MODELS_SCREEN_S8) + "_" + SCREENING_LIB_NAME_S8 + ".csv",
    sep=",", index=True
)

n_total_lib    = len(lib_raw_s8)
n_screened_lib = len(lib_clean_s8)
print(f"  Compounds in library        : {n_total_lib}")
print(f"  Compounds screened (no NaN) : {n_screened_lib}")
print(f"  Compounds excluded (NaN)    : {n_total_lib - n_screened_lib}")

# Compute individual model scores on the screening library
all_scores_s8 = []
for i, model_name in enumerate(model_names_s8):
    X_train = training_s8[best_feats_s8[i]]
    y_train = training_s8["clase"]
    reg_s8  = LinearRegression(fit_intercept=True, copy_X=True, n_jobs=None)
    reg_s8.fit(X_train, y_train)
    X_lib   = lib_clean_s8[best_feats_s8[i]]
    all_scores_s8.append(list(reg_s8.predict(X_lib)))

score_table_s8 = pd.DataFrame(all_scores_s8).T
score_table_s8.columns = model_names_s8
score_table_s8.index   = lib_clean_s8.index.tolist()

if "MIN" in ENSEMBLE_STRATEGY_S8:
    t = score_table_s8[model_names_s8[:N_MODELS_SCREEN_S8]]
    result_s8 = pd.concat([t, t.min(1)], axis=1).rename(
        columns={0: ENSEMBLE_STRATEGY_S8[0] + str(N_MODELS_SCREEN_S8)}
    )
    result_s8.to_csv(
        directorio_s8 + "/" + "screening_" + ENSEMBLE_STRATEGY_S8[0]
        + "_" + str(N_MODELS_SCREEN_S8) + "_" + SCREENING_LIB_NAME_S8 + ".csv",
        sep=",", index=True
    )

if "MEAN" in ENSEMBLE_STRATEGY_S8:
    t = score_table_s8[model_names_s8[:N_MODELS_SCREEN_S8]]
    result_s8 = pd.concat([t, t.mean(1)], axis=1).rename(
        columns={0: ENSEMBLE_STRATEGY_S8[0] + str(N_MODELS_SCREEN_S8)}
    )
    result_s8.to_csv(
        directorio_s8 + "/" + "screening_" + ENSEMBLE_STRATEGY_S8[0]
        + "_" + str(N_MODELS_SCREEN_S8) + "_" + SCREENING_LIB_NAME_S8 + ".csv",
        sep=",", index=True
    )

print("Step 9 complete.\n")


---

## Step 10 — Applicability Domain Assessment (Leverage / Hat-Matrix Rule)

Determines whether each compound in the screening library falls within the **applicability domain (AD)** of each ensemble model using the leverage (hat) matrix approach.

**Inputs:** File generated in the step 9 (eg: screening_xx_NUM_DrugBank). Select type ensemble and number of individuals models.
**Outputs:** AD boolean table (CSV) for all compounds and all models.


In [ ]:
print("=" * 70)
print("STEP 10 — Applicability domain (leverage rule)")
print("=" * 70)

# ----- USER CONFIGURATION ----------------------------------------------------
WORKING_DIR_S9         = Path(r"/Users/francocaram/Library/CloudStorage/GoogleDrive-francocaram8@gmail.com/My Drive/DOCTORADO/Cdifficile_ML/Data_sets")
TARGET_NAME_S9         = "CDIFF"
ENSEMBLE_STRATEGY_S9   = ["MIN"]
N_MODELS_AD_S9         = 5
SCREENING_LIB_NAME_S9  = "DrugBank"
TRAINING_FILE_S9       = "training_set.csv" 
BEST_FEATURES_FILE_S9  = "selected_models.csv"
SCREENING_SCORES_FILE_S9 = "screening_MIN_5_DrugBank.csv"
LEVERAGE_MULTIPLIER_S9 = 3   # (2 = strict; 3 = lenient)
# ----- END CONFIGURATION -----------------------------------------------------

directorio_s9 = str(WORKING_DIR_S9)

training_s9 = pd.read_csv(directorio_s9 + "/" + TRAINING_FILE_S9,
                           sep=",", header="infer", index_col=None)
lib_s9 = pd.read_csv(
    directorio_s9 + "/" + "descriptor_values_" + ENSEMBLE_STRATEGY_S9[0]
    + "_" + str(N_MODELS_AD_S9) + "_" + SCREENING_LIB_NAME_S9 + ".csv",
    sep=",", index_col="NAME"
)

# Parse selected model names from the first row of the screening scores file
with open(directorio_s9 + "/" + SCREENING_SCORES_FILE_S9, "r") as sc_file:
    header_row = sc_file.readline().strip().split(",")
selected_model_names_s9 = header_row[1:-1]

# Parse descriptor lists from best-features file
with open(directorio_s9 + "/" + BEST_FEATURES_FILE_S9, "r") as bf_file:
    next(bf_file)
    raw_feats_s9 = []
    for line in bf_file:
        parts = line.strip().split(",")
        raw_feats_s9.append(list(filter(None, parts)))

model_desc_s9 = [list(filter(None, raw_feats_s9[i][2:-1])) for i in range(len(selected_model_names_s9))]

# Compute leverage values and compare against threshold h*
ad_results_s9 = []
for feats in model_desc_s9:
    X_train_s9   = training_s9[feats]
    XtX_inv      = np.linalg.inv(X_train_s9.T.dot(X_train_s9))
    X_lib_s9     = lib_s9[feats]
    XtX_lib_T    = X_lib_s9.T
    XtX_lib_T.reset_index(drop=True, inplace=True)
    hat_values   = np.diag(X_lib_s9.dot(XtX_inv).dot(XtX_lib_T))
    h_threshold  = LEVERAGE_MULTIPLIER_S9 * (len(feats) / len(X_train_s9))
    ad_results_s9.append(["True" if h < h_threshold else "False" for h in hat_values])

ad_df_s9 = pd.DataFrame(ad_results_s9).T
ad_df_s9.columns = selected_model_names_s9
ad_df_s9.index   = lib_s9.index.tolist()
ad_df_s9.to_csv(
    directorio_s9 + "/" + "AD_" + ENSEMBLE_STRATEGY_S9[0] + "_"
    + str(N_MODELS_AD_S9) + "_" + SCREENING_LIB_NAME_S9
    + "_h" + str(LEVERAGE_MULTIPLIER_S9) + ".csv",
    index=True
)

print("Step 10 complete.\n")


---

## Step 11 — Integration of AD Results and Screening Scores

For each compound in the screening library, identifies which model produced the minimum score, then checks whether that compound lies within the **applicability domain** of that specific model. Also computes the percentage of ensemble models for which the compound is within the AD.
**Input:** Select the same value of leverage multiplier than before script (2 or 3) in LEVERAGE_MULT_S10 
**Outputs:** final compound-level table with score, AD status, and AD coverage (CSV).


In [ ]:
print("=" * 70)
print("STEP 11 — AD and screening score integration")
print("=" * 70)

# ----- USER CONFIGURATION ----------------------------------------------------
WORKING_DIR_S10        = Path(r"/Users/francocaram/Library/CloudStorage/GoogleDrive-francocaram8@gmail.com/My Drive/DOCTORADO/Cdifficile_ML/Data_sets")
TARGET_NAME_S10        = "CDIFF"
ENSEMBLE_S10           = "MIN"
N_MODELS_INT_S10       = "5"
SCREENING_LIB_NAME_S10 = "DrugBank"
SCREENING_SCORES_S10   = "screening_MIN_5_DrugBank.csv"
LEVERAGE_MULT_S10      = 3
# ----- END CONFIGURATION -----------------------------------------------------

directorio_s10 = str(WORKING_DIR_S10)

# Build dictionaries: compound → minimum score and → index of the model giving minimum
score_file_s10 = open(directorio_s10 + "/" + SCREENING_SCORES_S10, "r")
next(score_file_s10)
min_score_dict_s10  = {}
min_model_idx_s10   = {}
for line in score_file_s10:
    parts   = line.strip().split(",")
    cmpd    = parts[0]
    scores  = [float(v) for v in parts[1:-1]]
    min_score_dict_s10[cmpd] = min(scores)
    min_model_idx_s10[cmpd]  = scores.index(min(scores))
score_file_s10.close()

# Match AD table: for each compound, retrieve AD flag for the min-scoring model
ad_filename_s10 = (
    directorio_s10 + "/" + "AD_" + ENSEMBLE_S10 + "_" + N_MODELS_INT_S10
    + "_" + SCREENING_LIB_NAME_S10 + "_h" + str(LEVERAGE_MULT_S10) + ".csv"
)
ad_file_s10 = open(ad_filename_s10, "r")
next(ad_file_s10)

out_filename_s10 = (
    directorio_s10 + "/" + "AD_FINAL_" + ENSEMBLE_S10 + "_" + N_MODELS_INT_S10
    + "_" + SCREENING_LIB_NAME_S10 + "_h" + str(LEVERAGE_MULT_S10) + ".csv"
)
out_file_s10 = open(out_filename_s10, "w")

for line in ad_file_s10:
    line  = line.strip().replace('"', "")
    parts = line.split(",")
    cmpd  = parts[0]
    flags = parts[1:]
    pct_in_ad = round(flags.count("True") / len(flags), 4) * 100
    if cmpd in min_model_idx_s10:
        min_idx   = min_model_idx_s10[cmpd]
        min_score = min_score_dict_s10[cmpd]
        out_file_s10.write(
            cmpd + "," + str(min_score) + "," + flags[min_idx] + "," + str(pct_in_ad) + "\n"
        )
ad_file_s10.close()
out_file_s10.close()

result_s10 = pd.read_csv(out_filename_s10, header=None).rename(columns={
    0: "NAME", 1: ENSEMBLE_S10 + "_" + N_MODELS_INT_S10,
    2: "AD_status", 3: "pct_models_in_AD"
})
result_s10.to_csv(out_filename_s10, index=False)

print("Step 11 complete.\n")


---

## Step 12 — Candidate Selection: Drug-Likeness, PAINS, Tanimoto, Murcko Scaffolds

Filters compounds that exceed the activity score cutoff and lie within the AD. For each candidate, computes:

- **Drug-likeness rules:** Lipinski Ro5, Ghose filter, Veber filter, Rule of 3, REOS filter, and drug-like filter
- **fMF** — fraction of molecular framework (promiscuity indicator)
- **PAINS flag** — Pan-Assay Interference Compounds (RDKit catalog)
- **Maximum Tanimoto similarity** to active compounds in the training set
- **Murcko scaffold comparison** with training set actives and promiscuous scaffolds
- Whether the compound is already present in the training or test sets

**Input:** Select the cutoff value based on the prevoiusly decision steps (step 8) in ACTIVITY_CUTOFF_S11 
**Inputs:** File generated in the step 11 (eg: DA_FINAL_XXX_XX_DrugBank_h3.csv). Select type ensemble and number of individuals models.
**Outputs:** detailed candidate table (CSV and XLSX).


In [ ]:
print("=" * 70)
print("STEP 12 — Candidate selection and characterization")
print("=" * 70)

# ----- USER CONFIGURATION ----------------------------------------------------
WORKING_DIR_S11        = Path(r"/Users/francocaram/Library/CloudStorage/GoogleDrive-francocaram8@gmail.com/My Drive/DOCTORADO/Cdifficile_ML/Data_sets")
TARGET_NAME_S11        = "CDIFF"
ENSEMBLE_S11           = "MIN"
N_MODELS_CAND_S11      = "5"
SCREENING_LIB_NAME_S11 = "DrugBank"

SDF_ACTIVES_S11        = "inhibitors_tested.sdf"
SDF_SCREENING_LIB_S11  = "drugbank.sdf"
SDF_INACTIVES_S11      = "non_inhibitors_tested.sdf"
DESC_FILE_S11          = "descr_drugbank.txt"  # Tab-separated
SDF_EVALUATED_S11      = "all_tested.sdf"
FINAL_RESULTS_FILE_S11 = "AD_FINAL_MIN_5_DrugBank_h3.csv"
ACTIVITY_CUTOFF_S11    = 0.747
# ----- END CONFIGURATION -----------------------------------------------------

from rdkit import Chem
from rdkit.Chem import Descriptors, AllChem, DataStructs, MolToSmiles
from rdkit.Chem.Scaffolds import MurckoScaffold
from rdkit.Chem.FilterCatalog import FilterCatalog, FilterCatalogParams
import openpyxl

directorio_s11 = str(WORKING_DIR_S11)

# PAINS catalog
pains_params = FilterCatalogParams()
pains_params.AddCatalog(FilterCatalogParams.FilterCatalogs.PAINS_A)
pains_params.AddCatalog(FilterCatalogParams.FilterCatalogs.PAINS_B)
pains_params.AddCatalog(FilterCatalogParams.FilterCatalogs.PAINS_C)
pains_catalog = FilterCatalog(pains_params)

# Load active training compounds
actives_supplier = list(Chem.SDMolSupplier(directorio_s11 + "/" + SDF_ACTIVES_S11))
fw_actives_s11, fp_actives_s11, smiles_actives_train_s11 = [], [], []
for mol in actives_supplier:
    if mol is None:
        continue
    core = MurckoScaffold.GetScaffoldForMol(mol)
    fw   = MurckoScaffold.MakeScaffoldGeneric(core)
    fw_actives_s11.append(Chem.MolToSmiles(fw))
    fp_actives_s11.append(AllChem.GetMorganFingerprintAsBitVect(mol, 2, nBits=1024))
    smiles_actives_train_s11.append(Chem.MolToSmiles(mol))
fw_actives_set_s11     = set(fw_actives_s11)
smiles_actives_set_s11 = set(smiles_actives_train_s11)

# Load inactive training compounds
inact_supplier = list(Chem.SDMolSupplier(directorio_s11 + "/" + SDF_INACTIVES_S11))
smiles_inactives_set_s11 = set(Chem.MolToSmiles(mol) for mol in inact_supplier if mol is not None)

# Load previously evaluated compounds
eval_supplier = list(Chem.SDMolSupplier(directorio_s11 + "/" + SDF_EVALUATED_S11))
smiles_evaluated_s11, dict_evaluated_s11 = [], {}
for i_ev, mol in enumerate(eval_supplier, start=1):
    if mol is None:
        continue
    sm = Chem.MolToSmiles(mol)
    smiles_evaluated_s11.append(sm)
    dict_evaluated_s11[sm] = "compound_" + str(i_ev)

# Identify active candidates in AD
final_results_s11 = pd.read_csv(directorio_s11 + "/" + FINAL_RESULTS_FILE_S11, index_col="NAME")
ens_col_s11    = ENSEMBLE_S11 + "_" + N_MODELS_CAND_S11
active_cands   = final_results_s11[final_results_s11[ens_col_s11] > ACTIVITY_CUTOFF_S11]
cands_in_ad    = active_cands[active_cands["AD_status"] == True]
active_names   = cands_in_ad.index.tolist()

desc_data_s11  = pd.read_csv(directorio_s11 + "/" + DESC_FILE_S11, sep="\t", index_col="NAME")
all_lib_names  = desc_data_s11.index.tolist()
active_indices = [all_lib_names.index(m) for m in all_lib_names if m in active_names]

# Promiscuous scaffold list (Hu & Bajorath, 2008)
promiscuous_smiles_s11 = [
    "C(C1CCCCC1)C1CCCCC1", "C1CCCCC1", "C(CC1CCCCC1)C1CCCCC1", "C1CCCC1",
    "C1CCC2CCCCC2C1", "C1CC2CCCCC2C1", "C1CCC(CC1)C1CCCCC1", "C(CC1CCCCC1)CC1CCCCC1",
    "C(CCC1CCCCC1)CC1CCCCC1", "C(C1CCCC1)C1CCCCC1",
]
fw_promiscuous_s11 = []
for sm in promiscuous_smiles_s11:
    mol_p = Chem.MolFromSmiles(sm)
    sc_p  = MurckoScaffold.MakeScaffoldGeneric(MurckoScaffold.GetScaffoldForMol(mol_p))
    fw_promiscuous_s11.append(Chem.MolToSmiles(sc_p))

# Screening library SDF
lib_supplier_s11 = list(Chem.SDMolSupplier(directorio_s11 + "/" + SDF_SCREENING_LIB_S11))

# Process candidates — track valid names separately to handle None molecules
valid_names = []
(smiles_out, in_dataset, scaffold_match, fp_screening_s11,
 lipinski_s11, ro3_s11, ghose_s11, veber_s11, reos_s11, drug_like_s11,
 fmf_s11, sc_promiscuous_s11, logp_s11, pains_s11, prev_eval_s11) = (
    [], [], [], [], [], [], [], [], [], [], [], [], [], [], []
)

for idx in active_indices:
    mol = lib_supplier_s11[idx]
    if mol is None:
        print(f"  [WARNING] Could not parse molecule at index {idx} — skipped.")
        continue

    sm = MolToSmiles(mol)
    smiles_out.append(sm)
    valid_names.append(all_lib_names[idx])

    core  = MurckoScaffold.GetScaffoldForMol(mol)
    fw    = MurckoScaffold.MakeScaffoldGeneric(core)
    fw_sm = Chem.MolToSmiles(fw)
    fp_screening_s11.append(AllChem.GetMorganFingerprintAsBitVect(mol, 2, nBits=1024))

    in_dataset.append(
        "ACTIVE" if sm in smiles_actives_set_s11
        else ("INACTIVE" if sm in smiles_inactives_set_s11 else "NOT_IN_DATASET")
    )
    prev_eval_s11.append(dict_evaluated_s11.get(sm, "NOT_EVALUATED"))
    scaffold_match.append("YES" if fw_sm in fw_actives_set_s11 else "NO")
    sc_promiscuous_s11.append("YES" if fw_sm in fw_promiscuous_s11 else "NO")

    mw      = Descriptors.ExactMolWt(mol)
    logp    = Descriptors.MolLogP(mol)
    hbd     = Descriptors.NumHDonors(mol)
    hba     = Descriptors.NumHAcceptors(mol)
    rb      = Descriptors.NumRotatableBonds(mol)
    natm    = Chem.rdchem.Mol.GetNumAtoms(mol)
    mr      = Chem.Crippen.MolMR(mol)
    tpsa    = Chem.QED.properties(mol).PSA
    fc      = Chem.rdmolops.GetFormalCharge(mol)
    nhvy    = Chem.rdchem.Mol.GetNumHeavyAtoms(mol)
    nrng    = Chem.rdMolDescriptors.CalcNumRings(mol)
    nhvy_fw = Chem.rdchem.Mol.GetNumHeavyAtoms(fw)

    logp_s11.append(logp)
    fmf_s11.append(nhvy_fw / nhvy)

    lipinski_s11.append(sum([mw <= 500, logp <= 5, hbd <= 5, hba <= 5]))
    ro3_s11.append("YES" if mw <= 300 and logp <= 3 and hbd <= 3 and hba <= 3 and rb <= 3 else "NO")
    ghose_s11.append("YES" if 160 <= mw <= 480 and -0.4 <= logp <= 5.6
                      and 20 <= natm <= 70 and 40 <= mr <= 130 else "NO")
    veber_s11.append("YES" if rb <= 10 and tpsa <= 140 else "NO")
    reos_s11.append("YES" if 200 <= mw <= 500 and -5 <= logp <= 5
                    and 0 <= hbd <= 5 and 0 <= hba <= 10
                    and -2 <= fc <= 2 and 0 <= rb <= 8
                    and 15 <= nhvy <= 50 else "NO")
    drug_like_s11.append("YES" if mw < 400 and nrng > 0 and rb < 5
                          and hbd <= 5 and hba <= 10 and logp < 5 else "NO")
    pains_s11.append(
        pains_catalog.GetFirstMatch(mol).GetDescription()
        if pains_catalog.HasMatch(mol) else "PASS"
    )

# Tanimoto similarity to training actives
tan_matrix_s11 = pd.DataFrame(
    [[DataStructs.TanimotoSimilarity(fp_a, fp_b) for fp_b in fp_actives_s11]
     for fp_a in fp_screening_s11]
)
max_tc_s11 = tan_matrix_s11.max(axis=1)

candidates_df_s11 = pd.DataFrame({
    "SMILES": smiles_out,
    "In_training_dataset": in_dataset,
    "Murcko_scaffold_match": scaffold_match,
    "Max_Tanimoto": max_tc_s11.values,
    "Lipinski_Ro5_violations": [4 - v for v in lipinski_s11],
    "Rule_of_3": ro3_s11,
    "Ghose_filter": ghose_s11,
    "Veber_filter": veber_s11,
    "REOS_filter": reos_s11,
    "Drug_like_filter": drug_like_s11,
    "fMF": fmf_s11,
    "Promiscuous_scaffold": sc_promiscuous_s11,
    "MolLogP": logp_s11,
    "PAINS": pains_s11,
    "Previously_evaluated": prev_eval_s11,
})
candidates_df_s11.index = valid_names
cands_in_ad_valid = cands_in_ad[cands_in_ad.index.isin(valid_names)].drop_duplicates()
final_df_s11 = pd.concat([cands_in_ad_valid.reset_index(drop=True),
                           candidates_df_s11.reset_index(drop=True)], axis=1)
final_df_s11.index = valid_names

out_base_s11 = directorio_s11 + "/" + "final_candidates_" + TARGET_NAME_S11 + "_" + SCREENING_LIB_NAME_S11
final_df_s11.to_csv(out_base_s11 + ".csv", header=True)
final_df_s11.to_excel(out_base_s11 + ".xlsx", header=True)

print("Step 12 complete.\n")

print("=" * 70)
print("PIPELINE COMPLETE.")
print("=" * 70)